#  Dataset: House Prices - Advanced Regression Techniques

---

##  Objetivo del dataset
- **Meta principal**: **Predecir el precio de venta de una casa (`SalePrice`)** en la ciudad de Ames (Iowa, EE.UU.).
- **Cómo se logra**: mediante modelos de **regresión múltiple** u otros algoritmos de Machine Learning, usando como insumo las características de la casa (tamaño, materiales, condiciones, ubicación, etc.).

---

##  Pasos para llegar al objetivo
1. **Comprensión del problema**  
   - Variable dependiente: `SalePrice`.
   - Variables independientes: características estructurales, de ubicación, materiales y extras.

2. **Exploración de los datos (EDA)**  
   - Revisar registros, valores nulos, tipos de datos.
   - Generar estadísticas descriptivas y distribuciones.

3. **Limpieza de datos**  
   - Imputar o eliminar valores faltantes (`NA`).
   - Tratar valores atípicos.
   - Agrupar categorías poco frecuentes.

4. **Codificación de variables categóricas**  
   - Usar *One Hot Encoding* o *Label Encoding*.

5. **Escalamiento / Normalización**  
   - Aplicar sobre variables numéricas cuando el modelo lo requiera.

6. **Selección de variables relevantes**  
   - Usar correlaciones, ANOVA o técnicas de *feature selection*.

7. **Construcción del modelo**  
   - Empezar con **Regresión Lineal Múltiple**.
  

8. **Evaluación del modelo**  
- **RMSE (Root Mean Squared Error):** mide el error promedio, penaliza más los errores grandes.  
- **MAE (Mean Absolute Error):** error promedio absoluto entre lo real y lo predicho.  
- **R² (Coeficiente de determinación):** indica qué tan bien el modelo explica la variabilidad (0 a 1).  
- **Validación cruzada:** divide los datos en partes para entrenar y probar el modelo de forma más robusta.  


9. **Interpretación de resultados**  
   - Identificar qué variables influyen más en el precio de venta.

---

##  Resumen de variables principales
El dataset contiene **79 variables**. Algunos ejemplos clave:

- **Lote**
  - `MSSubClass`: tipo de vivienda (1 piso, 2 pisos, dúplex, PUD).  
  - `MSZoning`: zonificación (residencial, comercial, agrícola).  
  - `LotFrontage`: longitud de fachada.  
  - `LotArea`: área del terreno.  
  - `LotShape`: forma del lote (regular, irregular).  
  - `LandContour`: topografía (plano, pendiente, depresión).  
  - `Neighborhood`: barrio donde se ubica la casa.  

- **Construcción**
  - `BldgType`: tipo de construcción (unifamiliar, dúplex, townhouse).  
  - `HouseStyle`: estilo (1 piso, 2 pisos, split-level).  
  - `OverallQual`: calidad general (1 = muy mala, 10 = excelente).  
  - `OverallCond`: estado general de la vivienda.  
  - `YearBuilt`: año de construcción.  
  - `YearRemodAdd`: año de remodelación.  
  - `RoofStyle` / `RoofMatl`: tipo y material del techo.  
  - `Exterior1st` / `Exterior2nd`: material exterior.  

- **Sótano**
  - `BsmtQual`, `BsmtCond`: calidad y estado del sótano.  
  - `BsmtExposure`: exposición (ventanas, walkout).  
  - `TotalBsmtSF`: área total del sótano.  

- **Habitabilidad**
  - `GrLivArea`: área habitable sobre el nivel del suelo.  
  - `FullBath`, `HalfBath`: baños.  
  - `Bedroom`, `Kitchen`: habitaciones y cocinas.  
  - `TotRmsAbvGrd`: total de cuartos.  

- **Extras**
  - `Fireplaces`, `FireplaceQu`: chimeneas.  
  - `GarageType`, `GarageCars`, `GarageArea`: características del garaje.  
  - `WoodDeckSF`, `OpenPorchSF`, `PoolArea`, `Fence`: áreas exteriores y amenidades.  

- **Venta**
  - `MoSold`, `YrSold`: mes y año de venta.  
  - `SaleType`: tipo de venta (convencional, cash, VA loan).  
  - `SaleCondition`: condiciones de la venta (normal, familiar, parcial, etc.).  

- **Target**
  - `SalePrice`: precio de venta de la vivienda (variable a predecir).

---

##  En resumen
El dataset busca **explicar y predecir el precio de las casas** usando información estructural, de ubicación, calidad de materiales, extras y condiciones de venta.


In [ ]:
#salida 
import pandas as pd

# Crear los datos
data = {
    "Id": [1461, 1462, 1463],
    "SalePrice": [169277.0524984, 187758.393988768, 183583.683569555]
}

# Convertir a DataFrame
df = pd.DataFrame(data)

# Guardar en to_csv
df.to_csv("predicciones.csv", index=False)


print("Archivo 'predicciones.xlsx' generado con éxito ✅")


In [ ]:



import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Tamaño del dataset de entrenamiento:", train.shape)
print("Tamaño del dataset de prueba:", test.shape)
train.head()


print("\nColumnas con valores nulos en TRAIN:")
print(train.isnull().sum()[train.isnull().sum() > 0].sort_values(ascending=False).head(15))

print("\nDescripción estadística:")
print(train["SalePrice"].describe())


plt.figure(figsize=(8, 4))
sns.histplot(train["SalePrice"], kde=True, color="blue")
plt.title("Distribución del Precio de Venta")
plt.show()


corr = train.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr[["SalePrice"]].sort_values(by="SalePrice", ascending=False), annot=True, cmap="coolwarm")
plt.title("Correlación de variables con SalePrice")
plt.show()


umbral = len(train) * 0.3
train = train.dropna(thresh=umbral, axis=1)
test = test.dropna(thresh=len(test) * 0.3, axis=1)


for col in train.columns:
    if train[col].dtype == "object":
        train[col].fillna(train[col].mode()[0], inplace=True)
    else:
        train[col].fillna(train[col].median(), inplace=True)

for col in test.columns:
    if test[col].dtype == "object":
        test[col].fillna(test[col].mode()[0], inplace=True)
    else:
        test[col].fillna(test[col].median(), inplace=True)

print("Valores nulos después de limpieza:", train.isnull().sum().sum())



cat_cols = train.select_dtypes(include="object").columns


le = LabelEncoder()
for col in cat_cols:
    train[col] = le.fit_transform(train[col].astype(str))
    if col in test.columns:
        test[col] = le.transform(test[col].astype(str))


X = train.drop(["SalePrice", "Id"], axis=1)
y = train["SalePrice"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_scaled = scaler.transform(test.drop(["Id"], axis=1))

X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


modelo = LinearRegression()
modelo.fit(X_train, y_train)


y_pred = modelo.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R²: {r2:.4f}")


cv_scores = cross_val_score(modelo, X_scaled, y, cv=5, scoring="r2")
print(f"R² promedio (cross-validation): {cv_scores.mean():.4f}")


predicciones = modelo.predict(test_scaled)


output = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": predicciones
})


output.to_csv("predicciones.csv", index=False)
output.to_excel("predicciones.xlsx", index=False)

print("✅ Archivo 'predicciones.csv' y 'predicciones.xlsx' generados con éxito.")


importances = pd.Series(abs(modelo.coef_), index=X.columns)
print("\nVariables más influyentes:")
print(importances.sort_values(ascending=False).head(10))
